# this is for constructing the AI prompt. Do NOT run all without understanding the code below or talking with Noah

In [ ]:
#Set up
from pathlib import Path
import json
import os
import sys
import time
import warnings
import numpy as np
import logging


import pandas as pd
from IPython.display import display

rng = np.random.default_rng(seed=1312)
random_texts = rng.integers(1, 1061, size=35)

test_texts = pd.read_csv("full_text.csv", dtype={"id": str}).iloc[random_texts]
print(len(test_texts), "texts")  # Check: 50 texts
test_texts["president"].value_counts()

ModuleNotFoundError: No module named 'dotenv'

In [26]:
warnings.filterwarnings("ignore", category=FutureWarning)  # a harmless warning from one of the libraries

pd.set_option("display.max_colwidth", 300)  # show more of each text

SUPPORTED_PYTHON = (3, 13)
if sys.version_info[:2] != SUPPORTED_PYTHON:
    print(
        "Setup note: this course is tested with Python 3.13; "
        f"you are running {sys.version.split()[0]}."
    )

def find_course_root():
    """Find the cloned repository when this notebook is running locally."""
    for folder in (Path.cwd(), *Path.cwd().parents):
        if (folder / "pyproject.toml").exists() and (folder / "data").is_dir():
            return folder
    return None

LOCAL_COURSE_ROOT = find_course_root()
COURSE_ROOT = LOCAL_COURSE_ROOT or Path.cwd()
DATA_DIR = (
    LOCAL_COURSE_ROOT / "data"
    if LOCAL_COURSE_ROOT
    else COURSE_ROOT / "compss211_data"
)
GENERATED_DIR = COURSE_ROOT / "generated"
DATA_DIR.mkdir(parents=True, exist_ok=True)
GENERATED_DIR.mkdir(parents=True, exist_ok=True)

DATA_BASE_URL = (
    "https://raw.githubusercontent.com/"
    "macss-berkeley/compss-211a/main/data"
)

def course_data_path(filename):
    """Use a local course file, or download it when running in Colab."""
    path = DATA_DIR / filename
    if not path.exists():
        from urllib.request import urlretrieve

        urlretrieve(f"{DATA_BASE_URL}/{filename}", path)
    return path

print(f"Python {sys.version.split()[0]} | data={DATA_DIR}")

Setup note: this course is tested with Python 3.13; you are running 3.14.7.
Python 3.14.7 | data=c:\Users\cgsta\Documents\Berkeley\Computing I\Pres speech\Miller-211A\ai_junk\compss211_data


In [28]:
# Supplied: connect to Gemini. The key is read from your .env file and never printed.
import logging

from dotenv import load_dotenv
from google import genai
from google.genai import errors, types

load_dotenv(COURSE_ROOT / ".env")
key = os.environ.get("GEMINI_API_KEY", "")
if not key or key.startswith("paste-"):
    raise ValueError("Add GEMINI_API_KEY to your .env file, then run this cell again.")

client = genai.Client(api_key=key)
GEMINI_MODEL = "gemini-3.5-flash-lite"  # use the replacement model if bCourses announces one

# Hide a warning that the google-genai package prints on every call. It does not mean anything went wrong.
logging.getLogger("google_genai.types").setLevel(logging.ERROR)
print("Connected to Gemini.")

ModuleNotFoundError: No module named 'dotenv'